In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "nuget: FsLexYacc.Runtime, 11.3.0"
#r "../Scanning/bin/Debug/net9.0/Scanning.dll"
open System
open System.IO
open LectureScanning
// Run python FSharp/tools/build.py from the repository root first.
open System.Net
open System.Text

# Converting $\LaTeX$ to <span style="font-variant:small-caps;">Html</span>

The purpose of the following exercise is to implement a translator from [$\LaTeX$](http://www.latex-project.org) to 
[MathML](https://www.tutorialspoint.com/mathml/index.htm).  $\LaTeX$ is a document markup language
that is especially well suited to present text that contains mathematical formulas.  MathML is the part of <span style="font-variant:small-caps;">Html</span> that deals with the representation of mathematical formulas.  As $\LaTeX$ provides a very rich
document markup language and we can only afford to spend a few hours on this exercise, we confine
ourselves to a small subset of $\LaTeX$.  The file `example.tex` contains some $\LaTeX$.  The goal of this exercise is to implement a translator that is able to transform this file into MathML.

We start with reading the file. 

In [ ]:
let data = File.ReadAllText("example.tex")

Now data contains the text that is stored in this file.

In [ ]:
printfn "%s" data

Let us look at the output file `example.pdf` that is produced if we run $\LaTeX$ on this file. 
If you are nor running <span style="font-variant:small-caps;">MacOS</span> you have to replace the command
`open` with the name of an executable that can open a `.pdf`-file.

In [ ]:
// The reference PDF is available at example.pdf; open it from the file browser.

Next, we open the file `example.html`.  The scanner we are going to implement has to write its output into this file.

In [ ]:
let outfile = StringBuilder()
let write text = outfile.Append(text: string) |> ignore

<hr style="height:4px;background-color:blue">
Below are some predefined functions that you can use to create the <span style="font-variant:small-caps;">Html</span> file.
<hr style="height:4px;background-color:blue">

The function `startHtml` writes the header of the <span style="font-variant:small-caps;">Html</span> file
and the opening `<body>` tag to the string builder opened above.

In [ ]:
let startHtml () = write "<!doctype html><html><head><meta charset='utf-8'><title>LaTeX example</title></head><body>\n"

The function `endHtml` writes the closing `</body>` and `</html>` tags.

In [ ]:
let endHtml () = write "</body></html>"

The function `startMathBlock` starts a math block.  This is useful for formulas enclosed in `$$`.  This type of formulas is displayed in a line by itself.

In [ ]:
let startMathBlock () = write "<math xmlns='http://www.w3.org/1998/Math/MathML' display='block'>"

The function `startMathInline` starts an <em style="color:blue">inline formula</em>, i.e. a formula enclosed in `$`.  Formulas of this type are part of the surrounding text.

In [ ]:
let startMathInline () = write "<math xmlns='http://www.w3.org/1998/Math/MathML' display='inline'>"

The function `endMath` ends a math block.

In [ ]:
let endMath () = write "</math>"

The functions `startSum` and `endSum` write code to display formulas involving sums.  For example, to display  the expression
$$ \sum\limits_{i=1}^n i^2 $$
we can use the following MathML:
```
<munderover>
<mo>&sum;</mo>
<mrow>
<mi>i</mi>
<mo>=</mo>
<mn>1</mn>
</mrow>
<mrow>
<mi>n</mi>
</mrow>
</munderover>
<msup>
<mi>i</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>

```

In [ ]:
let startSum () = write "<munderover><mo>∑</mo>"
let endSum () = write "</munderover>"

The functions `startSqrt` and `endSqrt` write code to display formulas involving square roots.  For example, to display  the expression
$$ \sqrt{a^2 + b^2} $$
we can use the following MathML:
```
<msqrt>
<mrow>
<msup>
<mi>a</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>
<mo>+</mo>
<msup>
<mi>b</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>
</mrow>
</msqrt>
```

In [ ]:
let startSqrt () = write "<msqrt>"
let endSqrt () = write "</msqrt>"

In order to write exponents we have to use the tag `<msup>`.  For example, the expression $a^2$ 
is equivalent to the following markup:
```
<msup>
<mi>a</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>
```
Note that the exponent is enclosed in `<mrow>` `</mrow>` tags.

In [ ]:
let startSuper () = write "<msup>"
let endSuper () = write "</msup>"

In order to write fractions we have to use the tag `<mfrac>`.  For example, the expression $\frac{1}{6}$ 
is equivalent to the following markup:
```
<mfrac>
<mrow>
<mn>1</mn>
</mrow>
<mrow>
<mn>6</mn>
</mrow>
</mfrac>
```
Note that both nominator and denominator are enclosed in `<mrow>` `</mrow>` tags.

In [ ]:
let startFraction () = write "<mfrac>"
let endFraction () = write "</mfrac>"

Arguments of functions like the square root or exponents have to be enclosed in pairs of `<mrow>` and `</mrow>` tags.  

In [ ]:
let startRow () = write "<mrow>"
let endRow () = write "</mrow>"

Variable names should be enclosed in pairs of `<mi>` and `</mi>` tags.  For example, the variable $x$ is displayed by the following MathML:
```
<mi>x</mi>
```

In [ ]:
let writeVar v = write ("<mi>" + WebUtility.HtmlEncode(v) + "</mi>")

Numbers should be enclosed in pairs of `<mn>` and `</mn>` tags.  For example, the number $6$ is displayed by the following MathML:
```
<mn>6</mn>
```

In [ ]:
let writeNumber n = write ("<mn>" + WebUtility.HtmlEncode(n) + "</mn>")

The symbol $\cdot$ is created by the following MathML:
```
<mo>&sdot;</mo>
```

In [ ]:
let writeTimes () = write "<mo>⋅</mo>"

Mathematical operators should be enclosed in pairs of `<mo>` and `</mo>` tags.  For example, the operator $+$ is displayed by the following MathML:
```
<mo>+</mo>
```

In [ ]:
let writeOperator op = write ("<mo>" + WebUtility.HtmlEncode(op) + "</mo>")

The symbol $\pi$ is created by the following MathML:
```
<mn>&pi;</mn>
```

In [ ]:
let writePi () = write "<mi>π</mi>"

The symbol $\leq$ is created by the following MathML:
```
<mn>&le;</mn>
```

In [ ]:
let writeLeq () = write "<mo>≤</mo>"

The symbol $\geq$ is created by the following MathML:
```
<mn>&ge;</mn>
```

In [ ]:
let writeGeq () = write "<mo>≥</mo>"

The function `writeAny` writes a single character unadorned to the output file.

In [ ]:
let writeAny text = write (WebUtility.HtmlEncode text)

The token stream is produced by FsLex. Its `.fsl` specification is shown below and built by `../Scanning/Scanning.fsproj`.

In [ ]:
File.ReadAllText("../Scanning/Latex.fsl") |> printfn "%s"

We have to declare all tokens below.  We will need tokens for the following parts of the $\LaTeX$ file:
 - The $\LaTeX$ file starts with the string `\documentclass{article}`.
 - Next, there is the string `\begin{document}` that starts the content.
 - The string &#92;`end{document}` ends the content.
 - The string `$$` starts and ends a formula that is displayed on a line by itself.
 - The string `$` starts and ends a formula that is displayed as part of the text.
 - The string `\sum\limits_\{` starts the definition of a sum.
 - The string `\sqrt\{` starts the definition of a square root.
 - The string `\frac\{` starts the definition of a fraction.
 - A variable taken to a power starts something like `a^{`.
 - $\vdots$

In [ ]:
let supportedCommands = ["sum"; "limits"; "sqrt"; "frac"; "cdot"; "pi"; "leq"; "geq"]
supportedCommands

A recursive `row` function reads until a matching delimiter or closing brace. `atom` handles a formula component and an optional exponent; this preserves nesting without mutable untyped lexer attributes.

In [ ]:
// The recursive translator keeps group nesting on the F# call stack.

In [ ]:
let documentClass = "\\documentclass{article}"

In [ ]:
let beginDocument = "\\begin{document}"

In [ ]:
let endDocument = "\\end{document}"

In [ ]:
let blockDelimiter = "$$"

In [ ]:
let inlineDelimiter = "$"

In [ ]:
let echoText text = WebUtility.HtmlEncode text

In [ ]:
// Matching $$ closes a display formula; a different delimiter is rejected.

In [ ]:
// Matching $ closes an inline formula; a different delimiter is rejected.

## Completing the translation

The following recursive descent over FsLex tokens implements all listed formulas, including nested roots, fractions, exponents, sum limits and inline/display formulas. The F# call stack replaces the untyped scanner stack. Native MathML keeps the output usable offline.

In [ ]:
// Unknown commands and unmatched braces raise descriptive errors.

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

The line below generates the scanner.

In [ ]:
let tokenize = Scanning.latex

Next, we feed our input string into the generated scanner.

In [ ]:
let tokens = tokenize data

In order to scan the data that we provided in the last line, we iterate over all tokens generated by our scanner.

In [ ]:
let convertLatex (source: string) =
    let beginMarker = "\\begin{document}"
    let endMarker = "\\end{document}"
    let startIndex = source.IndexOf(beginMarker, StringComparison.Ordinal)
    let endIndex = source.IndexOf(endMarker, StringComparison.Ordinal)
    if startIndex < 0 || endIndex < startIndex then invalidArg "source" "Expected a complete document"
    let body = source.Substring(startIndex + beginMarker.Length, endIndex - startIndex - beginMarker.Length)
    let tokens = Scanning.latex body |> List.toArray
    let mutable position = 0
    let peek () = if position < tokens.Length then Some tokens.[position] else None
    let take () =
        match peek () with
        | Some token -> position <- position + 1; token
        | None -> failwith "Unexpected end of formula"
    let tag name text = "<" + name + ">" + text + "</" + name + ">"
    let rec row terminator : string =
        let output = StringBuilder()
        let mutable finished = false
        while not finished do
            match peek () with
            | Some token when token = terminator -> take () |> ignore; finished <- true
            | None -> failwith "Unclosed formula or group"
            | Some (Text text) when String.IsNullOrWhiteSpace text -> take () |> ignore
            | Some (MathDelimiter _) -> failwith "Mismatched math delimiter"
            | _ -> output.Append(atom ()) |> ignore
        tag "mrow" (output.ToString())
    and group () : string =
        match take () with
        | Brace '{' -> row (Brace '}')
        | token -> failwithf "Expected opening brace, found %A" token
    and atom () : string =
        let result =
            match take () with
            | Command "sqrt" -> tag "msqrt" (group ())
            | Command "frac" -> let numerator = group () in let denominator = group () in tag "mfrac" (numerator + denominator)
            | Command "sum" ->
                if take () <> Command "limits" || take () <> Text "_" then failwith "Expected sum limits"
                let lower = group ()
                if take () <> Hat then failwith "Expected upper sum limit"
                let upper = group ()
                tag "munderover" ("<mo>∑</mo>" + lower + upper)
            | Command "cdot" -> "<mo>⋅</mo>"
            | Command "pi" -> "<mi>π</mi>"
            | Command "leq" -> "<mo>≤</mo>"
            | Command "geq" -> "<mo>≥</mo>"
            | Command command -> failwithf "Unsupported LaTeX command: %s" command
            | Brace '{' -> row (Brace '}')
            | Text text ->
                let character = text.[0]
                let name = if Char.IsLetter character then "mi" elif Char.IsDigit character then "mn" else "mo"
                // Combine adjacent digits into one number. Variables remain individual symbols.
                let value = StringBuilder(text)
                if name = "mn" then
                    let mutable consuming = true
                    while consuming do
                        match peek () with
                        | Some(Text digit) when digit.Length = 1 && Char.IsDigit digit.[0] -> take () |> ignore; value.Append(digit) |> ignore
                        | _ -> consuming <- false
                tag name (WebUtility.HtmlEncode(value.ToString()))
            | token -> failwithf "Unexpected formula token: %A" token
        match peek () with
        | Some Hat -> take () |> ignore; tag "msup" (result + group ())
        | _ -> result
    let html = StringBuilder("<!doctype html><html><head><meta charset=\"utf-8\"><title>MathML</title></head><body>")
    while position < tokens.Length do
        match take () with
        | MathDelimiter delimiter ->
            let display = if delimiter = "$$" then "block" else "inline"
            html.Append("<math xmlns=\"http://www.w3.org/1998/Math/MathML\" display=\"" + display + "\">") |> ignore
            html.Append(row (MathDelimiter delimiter)).Append("</math>") |> ignore
        | Text text -> html.Append(WebUtility.HtmlEncode text) |> ignore
        | token -> failwithf "Unexpected token outside formula: %A" token
    html.Append("</body></html>").ToString()

In [ ]:
let html = convertLatex data
printfn "%s" html

In [ ]:
File.WriteAllText("example.html", html)

In [ ]:
checkNotebook (html.Contains("<mfrac>") && html.Contains("<msqrt>") && html.Contains("<munderover>"))
// Open example.html from the file browser; MathML needs no CDN or network.

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(html), "text/html") |> ignore